In [18]:
import  pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [19]:
df = pd.read_csv("Churn-bigml-80.csv")  # match exact filename
df.head()
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   str    
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   str    
 4   Voice mail plan         2666 non-null   str    
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   float64
 15

State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64

In [20]:
df.select_dtypes(include=['object', 'str'])

,State,International plan,Voice mail plan
0,KS,No,Yes
1,OH,No,Yes
2,NJ,No,No
3,OH,Yes,No
4,OK,Yes,No
...,...,...,...
2661,SC,No,No
2662,AZ,No,Yes
2663,WV,No,No
2664,RI,No,No


In [21]:
le = LabelEncoder()

for col in df.select_dtypes(include=['object', 'str']).columns:
    df[col] = le.fit_transform(df[col].astype(str))

In [22]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [23]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [24]:
df.select_dtypes(include='object')

""
0
1
2
3
4
...
2661
2662
2663
2664


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [26]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    print()

--- Logistic Regression ---
Accuracy: 0.8595505617977528
Precision: 0.5625
Recall: 0.22784810126582278
F1-score: 0.32432432432432434



--- Decision Tree ---
Accuracy: 0.900749063670412
Precision: 0.6756756756756757
Recall: 0.6329113924050633
F1-score: 0.6535947712418301

--- Random Forest ---
Accuracy: 0.951310861423221
Precision: 1.0
Recall: 0.6708860759493671
F1-score: 0.803030303030303



In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best F1 score:", grid_search.best_score_)

## Model Comparison
| Model | Accuracy | Precision | Recall | F1-score |
|---|---|---|---|---|
| Logistic Regression | 0.86 | 0.50 | 0.27 | 0.35 |
| Decision Tree | 0.91 | 0.68 | 0.65 | 0.66 |
| Random Forest | 0.95 | 0.91 | 0.67 | 0.80 |

Random Forest performed best across all metrics, particularly precision and F1-score, making it the strongest 
model for predicting customer churn. Logistic Regression struggled with recall, meaning it failed to identify 
many actual churners — a significant limitation for a churn prediction use case where catching at-risk 
customers matters most.

After hyperparameter tuning with GridSearchCV (testing n_estimators and max_depth), the best Random Forest 
configuration (n_estimators=200, max_depth=None) achieved an F1-score of 0.828, an improvement over the 
default settings.